# 09 — Forward-NaN Bisection (xformers / library root cause)

nb08 proved the v5 init weights are byte-perfect (0 missing, encoder/decoder/bias match the
file) yet `forward` returns **NaN** — so this is a RUNTIME/library failure, not a weight, save,
or projection failure. nb04 (which installed xformers) got a finite 7.390 on the SAME artifact;
nb08 (no xformers) got NaN.

NeoBERT's `model.py` hard-imports `from xformers.ops import SwiGLU` (used in every FFN block)
and uses torch `scaled_dot_product_attention` + a custom rotary. Our `salt3_common` patch only
swaps in a pure-torch SwiGLU when xformers is **absent** — if a leftover xformers is present but
ABI-incompatible with torch 2.11/cu128, the broken fused kernel runs and NaNs silently.

**This notebook:** (A) report env; (B) does BASE NeoBERT also NaN? (control: env vs our init);
(C) hook every module, find the FIRST op that emits NaN; (D) isolate SwiGLU vs attention;
(E) force pure-torch SwiGLU and re-test — if finite, xformers SwiGLU is the culprit and the fix
is to stop using it.


In [ ]:
%%capture
# match nb08's stack on purpose (do NOT install xformers here); we test what's present.
!pip install -U transformers safetensors huggingface_hub sentencepiece accelerate


In [ ]:
import sys, math
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', DEVICE)
try:
    import xformers, xformers.ops
    print('xformers PRESENT:', xformers.__version__, '-> NeoBERT will use its fused SwiGLU')
    XF = True
except Exception as e:
    print('xformers ABSENT ->', type(e).__name__, '(patched pure-torch SwiGLU should be used)')
    XF = False
try:
    import flash_attn; print('flash_attn:', flash_attn.__version__)
except Exception:
    print('flash_attn absent (fine; SDPA path used)')

INIT_DIR = PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias' / 'model'


## A. A reusable finite-forward probe + a Vietnamese batch


In [ ]:
def make_batch(tok, n=4):
    sents = ['Việt Nam là một quốc gia ở Đông Nam Á.',
             'Hôm nay thời tiết rất đẹp và trời trong xanh.',
             'Kinh tế Việt Nam tăng trưởng trong năm qua.',
             'Cô ấy đọc sách trong thư viện mỗi chiều.'][:n]
    return tok(sents, padding=True, truncation=True, max_length=32, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def forward_finite(model, enc):
    out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    t = out.logits if hasattr(out, 'logits') else out.last_hidden_state
    return bool(torch.isfinite(t).all()), t


## B. Control — does BASE NeoBERT (unmodified) also NaN here?
If base NeoBERT NaNs too, the fault is the ENVIRONMENT, not our init.


In [ ]:
base = AutoModelForMaskedLM.from_pretrained('chandar-lab/NeoBERT', trust_remote_code=True).to(DEVICE).eval()
base_tok = AutoTokenizer.from_pretrained('chandar-lab/NeoBERT', trust_remote_code=True)
print('base NeoBERT ffn type:', type(base.model.transformer_encoder[0].ffn).__module__ + '.' +
      type(base.model.transformer_encoder[0].ffn).__name__)
ok_base, t_base = forward_finite(base, make_batch(base_tok))
print(f'BASE NeoBERT forward finite: {ok_base}'
      + ('' if ok_base else '  <-- environment-level bug, independent of our init'))


## C. Our init forward + module-level NaN bisection
Hooks fire in execution order, so the first recorded module is where NaN is born.


In [ ]:
model = AutoModelForMaskedLM.from_pretrained(INIT_DIR, trust_remote_code=True).to(DEVICE).eval()
tok = AutoTokenizer.from_pretrained(INIT_DIR, trust_remote_code=True)
print('init ffn type:', type(model.model.transformer_encoder[0].ffn).__module__ + '.' +
      type(model.model.transformer_encoder[0].ffn).__name__)

first_nan = []
def mk(name):
    def hook(mod, inp, out):
        t = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(t) and not torch.isfinite(t).all():
            first_nan.append(name)
    return hook
handles = [m.register_forward_hook(mk(n)) for n, m in model.named_modules() if n]
ok_init, t_init = forward_finite(model, make_batch(tok))
for h in handles: h.remove()
print(f'init forward finite: {ok_init}')
print(f'first 8 modules emitting NaN (execution order): {first_nan[:8]}')
print(f'  -> NaN is born in: {first_nan[0] if first_nan else "(none — finite)"}')


## D. Isolate SwiGLU vs attention on the first encoder block
Feed a clean finite input straight into one block's FFN and attention to see which NaNs.


In [ ]:
blk = model.model.transformer_encoder[0]
x = torch.randn(2, 8, model.config.hidden_size, device=DEVICE)
with torch.no_grad():
    ffn_out = blk.ffn(x)
print(f'SwiGLU(ffn) on clean input finite: {bool(torch.isfinite(ffn_out).all())}  '
      f'(type {type(blk.ffn).__name__})')
# attention via a full block forward needs freqs_cis/mask; test SDPA primitive directly
q = torch.randn(2, 4, 8, 16, device=DEVICE); k = torch.randn_like(q); v = torch.randn_like(q)
sdpa = F.scaled_dot_product_attention(q.transpose(1,2), k.transpose(1,2), v.transpose(1,2),
                                      attn_mask=torch.ones(2,4,8,8, device=DEVICE).bool())
print(f'SDPA primitive finite: {bool(torch.isfinite(sdpa).all())}')


## E. Fix test — force pure-torch SwiGLU, re-run forward
If swapping the FFN to a pure-torch SwiGLU (same w12/w3 weights) makes forward finite, the
xformers fused SwiGLU is the culprit and the fix is to never use it.


In [ ]:
class PureSwiGLU(nn.Module):
    def __init__(self, w12, w3):
        super().__init__()
        self.w12, self.w3 = w12, w3   # reuse the loaded Linear layers (weights preserved)
    def forward(self, x):
        x1, x2 = self.w12(x).chunk(2, dim=-1)
        return self.w3(F.silu(x1) * x2)

swapped = 0
for blk in model.model.transformer_encoder:
    ffn = blk.ffn
    if hasattr(ffn, 'w12') and hasattr(ffn, 'w3'):
        blk.ffn = PureSwiGLU(ffn.w12, ffn.w3); swapped += 1
print(f'swapped {swapped} FFN blocks to pure-torch SwiGLU')
ok_fix, t_fix = forward_finite(model, make_batch(tok))
loss = float('nan')
if ok_fix:
    enc = make_batch(tok); ids = enc['input_ids']
    import torch as T; T.manual_seed(0)
    pm = T.full(ids.shape, 0.2);
    for sid in set(tok.all_special_ids): pm[ids.cpu()==sid]=0
    msk = T.bernoulli(pm).bool().to(DEVICE); lab = T.full_like(ids,-100); lab[msk]=ids[msk]
    mids = ids.clone(); mids[msk]=tok.mask_token_id
    with T.no_grad():
        lg = model(input_ids=mids, attention_mask=enc['attention_mask']).logits
    loss = F.cross_entropy(lg.reshape(-1,lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
print(f'forward finite AFTER pure-SwiGLU swap: {ok_fix}  | step-0 MLM loss {loss:.3f}')


## F. Verdict

In [ ]:
print('=' * 68); print('FORWARD-NAN BISECTION — VERDICT'); print('=' * 68)
print(f'env: torch {torch.__version__} | xformers {"present" if XF else "absent"}')
print(f'BASE NeoBERT forward finite : {ok_base}')
print(f'our init forward finite     : {ok_init}   first NaN @ {first_nan[0] if first_nan else "n/a"}')
print(f'after pure-SwiGLU swap      : {ok_fix}')
print('-' * 68)
if not ok_init and ok_fix:
    print('ROOT CAUSE: the xformers fused SwiGLU NaNs under this torch/CUDA. Fix = force the')
    print('pure-torch SwiGLU ALWAYS (do not gate on import success). Patch salt3_common to')
    print('override xformers.ops.SwiGLU even when importable, OR pin a matching xformers.')
elif not ok_init and not ok_fix:
    print('NaN is NOT (only) SwiGLU -> first-NaN module above points at attention/rotary/norm;')
    print('investigate that op under torch 2.11. (base finite=%s tells env-vs-init.)' % ok_base)
elif ok_init:
    print('Forward is finite in THIS run -> the nb08 NaN was environment state (leftover broken')
    print('xformers). Lock the stack: pin xformers to the torch build or always use fallback.')
print('=' * 68)
